In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


In [5]:
from pathlib import Path

In [6]:
import pandas as pd

In [7]:
project_dir = Path().resolve()

In [8]:
data_path = project_dir/"Data/household_energy_consumption.csv"

In [9]:
df = pd.read_csv(data_path,parse_dates=['Date'])

In [10]:
df.dtypes

Date                      datetime64[ns]
Energy_Consumption_kWh           float64
Household_Size                     int64
Avg_Temperature_C_x              float64
Has_AC                             int64
Peak_Hours_Usage_kWh             float64
Avg_Temperature_C_y              float64
Avg_Humidity_Pct                   int64
Day                                int64
Month                              int64
Is Weekend                         int64
Is_Holiday                         int64
dtype: object

In [11]:
df.sort_values(by='Date')

,Date,Energy_Consumption_kWh,Household_Size,Avg_Temperature_C_x,Has_AC,Peak_Hours_Usage_kWh,Avg_Temperature_C_y,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
0,2025-01-04,8.4,4,17.8,0,3.2,12.0,80,5,1,1,0
8565,2025-01-04,7.6,3,18.2,0,2.7,12.0,80,5,1,1,0
8566,2025-01-04,18.8,5,16.8,1,7.8,12.0,80,5,1,1,0
8567,2025-01-04,12.0,5,14.7,0,3.8,12.0,80,5,1,1,0
8568,2025-01-04,4.4,2,15.9,0,1.5,12.0,80,5,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
78389,2025-07-04,7.4,3,14.7,0,2.8,19.6,75,4,7,0,1
78390,2025-07-04,5.9,2,14.2,0,2.1,19.6,75,4,7,0,1
78391,2025-07-04,15.1,6,18.1,0,5.9,19.6,75,4,7,0,1
78379,2025-07-04,7.0,3,17.8,0,2.5,19.6,75,4,7,0,1


In [12]:
df.isnull().sum()

Date                      0
Energy_Consumption_kWh    0
Household_Size            0
Avg_Temperature_C_x       0
Has_AC                    0
Peak_Hours_Usage_kWh      0
Avg_Temperature_C_y       0
Avg_Humidity_Pct          0
Day                       0
Month                     0
Is Weekend                0
Is_Holiday                0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(2570)

In [14]:
duplicates = df[df.duplicated()]

In [15]:
len(duplicates)/len(df)

0.03177782723743106

In [16]:
duplicates.sort_values(by='Energy_Consumption_kWh',ascending=False)

,Date,Energy_Consumption_kWh,Household_Size,Avg_Temperature_C_x,Has_AC,Peak_Hours_Usage_kWh,Avg_Temperature_C_y,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
80761,2025-07-04,20.0,6,18.4,1,9.7,19.6,75,4,7,0,1
80549,2025-07-04,20.0,5,15.7,1,8.2,19.6,75,4,7,0,1
400,2025-01-04,20.0,6,18.1,1,9.9,12.0,80,5,1,1,0
972,2025-01-04,20.0,6,16.6,1,8.9,12.0,80,5,1,1,0
80400,2025-07-04,20.0,6,18.0,1,9.4,19.6,75,4,7,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2907,2025-01-04,0.5,1,17.4,0,0.2,12.0,80,5,1,1,0
62744,2025-05-04,0.5,1,13.9,0,0.2,14.3,78,6,5,1,0
1915,2025-01-04,0.5,1,18.2,0,0.2,12.0,80,5,1,1,0
2587,2025-01-04,0.5,1,15.2,0,0.2,12.0,80,5,1,1,0


In [17]:
(df.drop_duplicates(inplace=True,ignore_index=True))

In [18]:
df.drop(columns=['Date','Energy_Consumption_kWh','Avg_Temperature_C_y'],inplace=True)

In [19]:
df

,Household_Size,Avg_Temperature_C_x,Has_AC,Peak_Hours_Usage_kWh,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
0,4,17.8,0,3.2,80,5,1,1,0
1,5,20.1,1,7.8,80,5,1,1,0
2,3,12.7,0,3.1,80,5,1,1,0
3,5,15.1,1,7.2,80,5,1,1,0
4,2,21.9,0,1.8,80,5,1,1,0
...,...,...,...,...,...,...,...,...,...
78299,4,14.9,1,6.4,75,4,7,0,1
78300,5,17.5,1,8.1,75,4,7,0,1
78301,4,16.9,1,6.3,75,4,7,0,1
78302,4,18.2,1,6.6,75,4,7,0,1


In [20]:
splix_idx=int(len(df)*0.8)

In [21]:
splix_idx

62643

In [22]:
features = df.drop(columns=['Peak_Hours_Usage_kWh'])

In [23]:
features

,Household_Size,Avg_Temperature_C_x,Has_AC,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
0,4,17.8,0,80,5,1,1,0
1,5,20.1,1,80,5,1,1,0
2,3,12.7,0,80,5,1,1,0
3,5,15.1,1,80,5,1,1,0
4,2,21.9,0,80,5,1,1,0
...,...,...,...,...,...,...,...,...
78299,4,14.9,1,75,4,7,0,1
78300,5,17.5,1,75,4,7,0,1
78301,4,16.9,1,75,4,7,0,1
78302,4,18.2,1,75,4,7,0,1


In [24]:
target = features = df['Peak_Hours_Usage_kWh']

In [25]:
def load_data(path):
    """Loading the data and Processing"""
    df = pd.read_csv(path,parse_dates=['Date'])
    df.drop_duplicates(inplace=True,ignore_index=True)
    df.drop(columns=['Date','Energy_Consumption_kWh','Avg_Temperature_C_y'],inplace=True)
    features = df.drop(columns=['Peak_Hours_Usage_kWh'])
    target = df['Peak_Hours_Usage_kWh']
    splix_idx=int(len(df)*0.8)
    X_train = features.iloc[:splix_idx]
    y_train = target.iloc[:splix_idx]
    X_test = features.iloc[splix_idx:]
    y_test = target.iloc[splix_idx:]
    return {
        "X_train":X_train,
        "y_train":y_train,
        "X_test":X_test,
        "y_test":y_test,
        "features":features,
        "target":target
    }

In [26]:
bundled_data=load_data(data_path)

In [27]:
X_train= bundled_data['X_train']
y_train = bundled_data['y_train']
X_test= bundled_data['X_test']
y_test = bundled_data['y_test']
features = bundled_data['features']
target=bundled_data['target']

In [28]:
X_train

,Household_Size,Avg_Temperature_C_x,Has_AC,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
0,4,17.8,0,80,5,1,1,0
1,5,20.1,1,80,5,1,1,0
2,3,12.7,0,80,5,1,1,0
3,5,15.1,1,80,5,1,1,0
4,2,21.9,0,80,5,1,1,0
...,...,...,...,...,...,...,...,...
62638,4,17.4,1,78,2,6,0,0
62639,1,15.8,1,78,2,6,0,0
62640,3,20.1,0,78,2,6,0,0
62641,1,14.8,1,78,2,6,0,0


In [29]:
new_model = RandomForestRegressor(n_estimators=400, max_depth=3)

In [30]:
features

,Household_Size,Avg_Temperature_C_x,Has_AC,Avg_Humidity_Pct,Day,Month,Is Weekend,Is_Holiday
0,4,17.8,0,80,5,1,1,0
1,5,20.1,1,80,5,1,1,0
2,3,12.7,0,80,5,1,1,0
3,5,15.1,1,80,5,1,1,0
4,2,21.9,0,80,5,1,1,0
...,...,...,...,...,...,...,...,...
78299,4,14.9,1,75,4,7,0,1
78300,5,17.5,1,75,4,7,0,1
78301,4,16.9,1,75,4,7,0,1
78302,4,18.2,1,75,4,7,0,1


In [31]:
new_model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [32]:
new_predict = new_model.predict(X_test)

In [33]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [34]:
print(f"MSE {mean_squared_error (y_test,new_predict)}")
print(f"MAE {mean_absolute_error (y_test,new_predict)}")
print(f"r2 {r2_score (y_test,new_predict)}")
print(f"RMSE {root_mean_squared_error (y_test,new_predict)}")

MSE 0.6027520973344672
MAE 0.617511523848169
r2 0.9029798410954328
RMSE 0.7763711079982737


In [35]:
# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 200, 1000,steps=100),
#         "max_depth": trial.suggest_int("max_depth", 4, 10),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
#         "subsample": trial.suggest_float("subsample", 0.7, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
#         "gamma": trial.suggest_float("gamma", 0, 2),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0, 3),
#     }
#     model=XGBRegressor(**params)

#     scores = cross_val_score(model,X_train,y_train, cv=5,scoring="neg_root_mean_squared_error",n_jobs=-1)

#     return scores.mean()

In [36]:
import optuna
from sklearn.model_selection import cross_val_score

c:\Users\Shaaf\Desktop\Data Science\Practice Projects\DeepLearning\Energy Predictions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_samples_split": trial.suggest_float("min_samples_split", 0.7, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "random_state": 42,
        "n_jobs": -1
    }
    model=RandomForestRegressor(**params)

    scores = cross_val_score(model,X_train,y_train, cv=5,scoring="neg_root_mean_squared_error",n_jobs=-1)

    return scores.mean()

In [38]:
# study = optuna.create_study(direction="minimize")

# study.optimize(
#     objective,
#     n_trials=100,
#     show_progress_bar=True
# )

In [39]:
# print("Best RMSE:", study.best_value)

# print("Best Parameters:")
# print(study.best_params)

In [55]:
xgb_model = XGBRegressor(**grid_search.best_params_)

In [56]:
xgb_model.fit(X_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [45]:
from sklearn.model_selection import GridSearchCV

In [60]:
params={
    "n_estimators":[800,1000,1200,1600],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

In [61]:
model = XGBRegressor()

In [62]:
grid_search = GridSearchCV(estimator=model, param_grid=params,scoring="r2",cv=5, n_jobs=-1, verbose=1)

In [63]:
grid_search.fit(X_train,y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'n_estimators': [800, 1000, ...], 'subsample': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter 

In [65]:
grid_search.best_params_

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 800, 'subsample': 0.8}

In [57]:
xgb_predicted = xgb_model.predict(X_test)

In [58]:
print(f"MSE {mean_squared_error (y_test,xgb_predicted)}")
print(f"MAE {mean_absolute_error (y_test,xgb_predicted)}")
print(f"r2 {r2_score (y_test,xgb_predicted)}")
print(f"RMSE {root_mean_squared_error (y_test,xgb_predicted)}")

MSE 0.44930074884281823
MAE 0.5316283454734256
r2 0.9276796708938162
RMSE 0.6702989995836323
